# Problem 2 - Transformer from scratch

Run this notebook from top to bottom. Full mode writes samples, figures, and CSV evidence.

In [ ]:
from pathlib import Path
import sys
import subprocess
import torch
FAST_MODE = False
ROOT = Path.cwd().parent if Path.cwd().name == 'problem2' else Path.cwd()
subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", "main"], check=True)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'problem2'))
from problem2.problem2 import (
    hand_check_attention, plot_hand_check, run_har_transformer,
    run_transformer_ablations, hand_attention_table,
    run_mask_and_context_ablation, train_language_model,
    generate_samples, plot_attention_heatmaps,
    CausalTransformerLM,
)
import time
out, weights = hand_check_attention()
print(out.shape, weights.shape)
q_hand, k_hand, v_hand, scores, hand_weights, hand_output = hand_attention_table()
print('Q=', q_hand, 'K=', k_hand, 'V=', v_hand)
print('hand scores=', scores, 'hand weights=', hand_weights, 'hand output=', hand_output)
plot_hand_check(ROOT / 'problem2' / 'figures' / 'p2_hand_attention.png')
assert torch.allclose(weights.sum(-1), torch.ones(1, 3), atol=1e-6)
assert torch.allclose(hand_output, hand_weights @ v_hand, atol=1e-6)
from src.models import encoder_block_parameter_breakdown
breakdown = encoder_block_parameter_breakdown(128)
one_block = CausalTransformerLM(65, 128, 4, 1, 128)
print('Encoder weight derivation:', breakdown['formula'], breakdown['weights_only'])
print('One-block parameters:', sum(p.numel() for p in one_block.parameters()))
assert breakdown['weights_only'] == 196608


## Full HAR Transformer positional ablation

Runs all three seeds for no, sinusoidal, and learned positional information.

In [ ]:
started = time.perf_counter()
transformer_har_results = run_har_transformer(False)
print("Transformer HAR runtime seconds:", time.perf_counter() - started)
assert len(transformer_har_results) == 9


## Full head-count ablation

Measures one, four, and eight heads at fixed width.

In [ ]:
started = time.perf_counter()
head_results = run_transformer_ablations(False)
print("Head ablation runtime seconds:", time.perf_counter() - started)
assert len(head_results) == 3
assert len({row[2] for row in head_results}) == 1


## Full causal-mask and context-length study

Measures causal versus unmasked forward passes for contexts 64, 128, 256, and 512.

In [ ]:
started = time.perf_counter()
context_results = run_mask_and_context_ablation(False)
print("Context ablation runtime seconds:", time.perf_counter() - started)
assert len(context_results) == 4


## Full Tiny Shakespeare language model

Trains the causal character model and writes the validation curve and metrics.

In [ ]:
started = time.perf_counter()
model, chars, encode, lm_history = train_language_model(False)
print(lm_history[-1], lm_history[-1] / torch.log(torch.tensor(2.0)).item())
print("Language-model runtime seconds:", time.perf_counter() - started)
assert len(lm_history) == 12


## Required generated samples and attention heatmaps

Writes 500-character samples at temperatures 0.2, 0.8, and 1.5, then saves two attention heatmaps.

In [ ]:
generate_samples(model, chars, encode, ROOT / 'problem2' / 'generated_samples.txt')
prompt = 'First Citizen:\nBefore we proceed any further, hear me speak.'
tokens = torch.tensor([[encode[ch] for ch in prompt]], device=next(model.parameters()).device)
plot_attention_heatmaps(model, tokens, ROOT / 'problem2' / 'figures' / 'p2_attention_heatmaps.png')
print('samples and heatmap saved')